# Multiclass Classification with Spark Batch Trainer

This notebook compares **XGBoost**, **CatBoost**, and **LightGBM** on a seven-class obesity dataset. It demonstrates the complete multiclass workflow with the current package architecture and public API.

## Learning objectives

- inspect and validate a multiclass dataset;
- encode the target without leaking information from validation or test;
- create reproducible stratified splits;
- configure all three backends for multiclass objectives;
- compare class-level errors and probability quality; and
- interpret confidence without treating confidence as calibration.

> **Terminology:** this is multiclass classification—one target and one class per observation—not multilabel classification.

## Phase 1 — Environment and reproducibility

Resolve the repository root explicitly so the notebook behaves the same from the project root and the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Return the nearest parent containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml from the current directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from pyspark.sql import SparkSession
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from spark_batch_trainer import create_trainer

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
TARGET_COLUMN = "NObeyesdad"

## Phase 2 — Load and understand the dataset

Validate the target and basic schema before encoding labels. Class frequency matters because overall accuracy can hide weak performance on smaller classes.

In [ ]:
dataset_path = PROJECT_ROOT / "data" / "multiclass_dataset" / "ObesityDataset.csv"
dataset = pd.read_csv(dataset_path)

assert TARGET_COLUMN in dataset.columns, f"Missing target column: {TARGET_COLUMN}"
assert dataset[TARGET_COLUMN].nunique() > 2, "A multiclass target was expected."
assert not dataset.columns.duplicated().any(), "Duplicate column names detected."

quality_summary = pd.DataFrame(
    {
        "rows": [len(dataset)],
        "features": [dataset.shape[1] - 1],
        "classes": [dataset[TARGET_COLUMN].nunique()],
        "missing_values": [int(dataset.isna().sum().sum())],
        "duplicate_rows": [int(dataset.duplicated().sum())],
    }
)
display(quality_summary)
display(dataset.head())

In [ ]:
class_distribution = (
    dataset[TARGET_COLUMN]
    .value_counts()
    .rename_axis("class_name")
    .reset_index(name="rows")
)
class_distribution["share"] = class_distribution["rows"] / len(dataset)
display(class_distribution)

plt.figure(figsize=(10, 4))
sns.barplot(data=class_distribution, x="class_name", y="rows", color="#2563EB")
plt.title("Target-class distribution", fontweight="bold")
plt.xlabel("Class")
plt.ylabel("Rows")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Interpretation checkpoint

Review whether every class has enough observations for all three splits. If one class is rare, prefer balanced accuracy and per-class recall over overall accuracy alone.

## Phase 3 — Stratified splitting and target encoding

Split first, then fit the label encoder on the training target only. Transform validation and test with the same mapping so class identifiers remain stable.

In [ ]:
train_df, temporary_df = train_test_split(
    dataset,
    test_size=0.30,
    stratify=dataset[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

label_encoder = LabelEncoder().fit(train_df[TARGET_COLUMN])
for frame in (train_df, validation_df, test_df):
    frame.loc[:, TARGET_COLUMN] = label_encoder.transform(frame[TARGET_COLUMN])
    frame[TARGET_COLUMN] = frame[TARGET_COLUMN].astype(int)

CLASS_NAMES = label_encoder.classes_.tolist()
NUMBER_OF_CLASSES = len(CLASS_NAMES)
CLASS_IDS = list(range(NUMBER_OF_CLASSES))

mapping = pd.DataFrame({"class_id": CLASS_IDS, "class_name": CLASS_NAMES})
display(mapping)

In [ ]:
split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "classes": frame[TARGET_COLUMN].nunique(),
        }
        for name, frame in (
            ("train", train_df),
            ("validation", validation_df),
            ("test", test_df),
        )
    ]
)
assert split_summary["classes"].eq(NUMBER_OF_CLASSES).all()
display(split_summary)

## Phase 4 — Start Spark and create trainer inputs

The trainer receives Spark DataFrames. The untouched test set remains in pandas for final native-model evaluation.

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkBatchTrainerMulticlassGuide")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

spark_train_df = spark.createDataFrame(train_df)
spark_validation_df = spark.createDataFrame(validation_df)

print(f"Spark version: {spark.version}")
print(f"Training rows: {spark_train_df.count():,}")
print(f"Validation rows: {spark_validation_df.count():,}")

## Phase 5 — Shared controls and evaluation helpers

Multiclass log loss is minimized. Balanced accuracy is included in the final comparison to give every class equal importance.

In [ ]:
TRAINING_CONFIG = {
    "num_batches": 5,
    "max_patience": 3,
    "metric_mode": "min",
    "min_delta": 1e-4,
    "use_sample_weight": False,
    "show_learning_curve": False,
    "verbose": True,
}

LEARNING_RATE_CONFIG = {
    "initial_lr": 0.05,
    "decay_rate": 0.95,
    "min_lr": 0.005,
}

In [ ]:
def prepare_native_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Return features with categorical levels represented consistently."""
    features = frame.drop(columns=[TARGET_COLUMN]).copy()
    categorical_columns = features.select_dtypes(include=["object"]).columns
    features[categorical_columns] = features[categorical_columns].astype("category")
    return features


def evaluate_multiclass_model(name: str, model, frame: pd.DataFrame) -> dict:
    """Evaluate hard predictions and probability quality on one split."""
    features = prepare_native_features(frame)
    target = frame[TARGET_COLUMN].to_numpy(dtype=int)
    predictions = np.asarray(model.predict(features)).reshape(-1).astype(int)
    probabilities = np.asarray(model.predict_proba(features))
    return {
        "model": name,
        "accuracy": accuracy_score(target, predictions),
        "balanced_accuracy": balanced_accuracy_score(target, predictions),
        "log_loss": log_loss(target, probabilities, labels=CLASS_IDS),
        "predictions": predictions,
        "probabilities": probabilities,
    }


def summarize_history(trainer) -> pd.DataFrame:
    """Return the last train and validation metric recorded per batch."""
    history = trainer.get_training_history()
    return pd.DataFrame(
        {
            "batch": history.batch_numbers,
            "train_metric": [values[-1] for values in history.train_scores],
            "validation_metric": [
                values[-1] for values in history.validation_scores
            ],
        }
    )

## Phase 6 — Train XGBoost

`multi:softprob` produces class probabilities, which are required for multiclass log loss and confidence analysis.

In [ ]:
xgboost_trainer = create_trainer("xgboost")
xgboost_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "objective": "multi:softprob",
        "num_class": NUMBER_OF_CLASSES,
        "eval_metric": "mlogloss",
        "n_estimators": 50,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_STATE,
    },
    training_config=TRAINING_CONFIG,
    learning_rate_config=LEARNING_RATE_CONFIG,
)
xgboost_model = xgboost_trainer.get_trained_model()
display(summarize_history(xgboost_trainer))

## Phase 7 — Train CatBoost

CatBoost uses its native multiclass objective. File output is disabled to keep notebook runs free of generated training directories.

In [ ]:
catboost_trainer = create_trainer("catboost")
catboost_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 50,
        "learning_rate": 0.05,
        "depth": 6,
        "random_seed": RANDOM_STATE,
        "allow_writing_files": False,
        "verbose": False,
    },
    training_config=TRAINING_CONFIG,
)
catboost_model = catboost_trainer.get_trained_model()
display(summarize_history(catboost_trainer))

## Phase 8 — Train LightGBM

LightGBM requires both the multiclass objective and the number of encoded classes.

In [ ]:
lightgbm_trainer = create_trainer("lightgbm")
lightgbm_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "objective": "multiclass",
        "num_class": NUMBER_OF_CLASSES,
        "metric": "multi_logloss",
        "n_estimators": 50,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_STATE,
        "verbosity": -1,
    },
    training_config=TRAINING_CONFIG,
    learning_rate_config=LEARNING_RATE_CONFIG,
)
lightgbm_model = lightgbm_trainer.get_trained_model()
display(summarize_history(lightgbm_trainer))

## Phase 9 — Final model comparison

Evaluate the three selected models once on the test set. Lower log loss indicates better probability assignments; higher balanced accuracy indicates stronger average class recall.

In [ ]:
evaluation_results = [
    evaluate_multiclass_model("XGBoost", xgboost_model, test_df),
    evaluate_multiclass_model("CatBoost", catboost_model, test_df),
    evaluate_multiclass_model("LightGBM", lightgbm_model, test_df),
]

comparison = pd.DataFrame(evaluation_results).drop(
    columns=["predictions", "probabilities"]
)
display(comparison.sort_values("log_loss").reset_index(drop=True))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(20, 6))
for axis, result in zip(axes, evaluation_results):
    matrix = confusion_matrix(
        test_df[TARGET_COLUMN], result["predictions"], labels=CLASS_IDS, normalize="true"
    )
    sns.heatmap(
        matrix,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        cbar=False,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )
    axis.set_title(result["model"])
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.tick_params(axis="x", rotation=45)
plt.suptitle("Row-normalized multiclass confusion matrices", fontweight="bold")
plt.tight_layout()
plt.show()

## Phase 10 — Confidence diagnostics

Maximum predicted probability is a useful uncertainty signal, but it is not proof that probabilities are calibrated. Compare distributions across models and follow up with a dedicated calibration study when probabilities drive decisions.

In [ ]:
confidence_rows = []
for result in evaluation_results:
    confidence_rows.extend(
        {"model": result["model"], "maximum_probability": value}
        for value in result["probabilities"].max(axis=1)
    )
confidence_df = pd.DataFrame(confidence_rows)

plt.figure(figsize=(10, 5))
sns.histplot(
    data=confidence_df,
    x="maximum_probability",
    hue="model",
    bins=20,
    element="step",
    stat="density",
    common_norm=False,
)
plt.title("Test-set maximum probability by model", fontweight="bold")
plt.xlabel("Maximum predicted probability")
plt.tight_layout()
plt.show()

In [ ]:
best_result = min(evaluation_results, key=lambda item: item["log_loss"])
print(f"Detailed test report for {best_result['model']}:")
print(
    classification_report(
        test_df[TARGET_COLUMN],
        best_result["predictions"],
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

### Interpretation guidance

A strong aggregate score can coexist with weak recall for one class. Use the normalized confusion matrices and classification report to identify systematic class confusion. Treat this notebook as a transparent workflow baseline rather than a final model-selection study.

## Phase 11 — Resource cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")

## Next steps

A production study should add cross-validated hyperparameter selection, probability calibration on validation data, experiment tracking, native-model persistence, and explicit monitoring for class-distribution drift.